In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import numpy as np
import torch.optim as optim
import matplotlib as mpb
from matplotlib import pyplot as plt
from sklearn import metrics
import seaborn as sn
import pandas as pd
import photoncounter as pc

### DATA-GATHERING IS HERE
### DATA-GATHERING IS HERE
### DATA-GATHERING IS HERE
# Create statistical model of Xe scintillation (ER and NR)
e = pc.simulate
nr = pc.simulate.xenon_scintillation_time_generator(weighting_ratio=pc.simulate.NR_WEIGHTING_RATIO)
er = pc.simulate.xenon_scintillation_time_generator(weighting_ratio=pc.simulate.ER_WEIGHTING_RATIO)
# Create statistical model of LZ detector repsonse (photon time-of-flight)
tof = pc.simulate.lz_photon_detection_time_generator()
# Create a statistical model of the PMT reponse (anode charge)
pmt= pc.simulate.PMT()
npconfusion = np.zeros((0,7),dtype=int)
fancyconfusiontrue = [0,1,2,3,4,5,6]
fancyconfusionguess = [0,1,2,3,4,5,6]
events = 10
N_pmts = 2
ycml = []
sumtml = []
sumnpt = []
ybase = np.array([[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]])
nptbase = np.array([[0]])
# Get a photon template
m = pc.ExpGauss(0)
# The time axis over which to simulate
timeline = np.arange(-100, 400, 10) #  <= We have 100 MHz (10 ns) sampling in LZ.  it's slower than you think
timeline_highres = np.arange(-100, 400, 0.1)
confarray = [[0,0,0,0],
        [0,0,0,0],
        [0,0,0,0],
        [0,0,0,0]]
counter = 0
wincounter = 0
zcount = 0
zwin = 0
ocount = 0
owin = 0
tcount = 0
twin = 0
hcount = 0
hwin = 0
for i in range(0,1000):
    evt = 100
    npt = random.randint(0,7)
    nr_times = nr.sample((evt, npt))
    er_times = er.sample((evt, npt))
    det_times = tof.sample((evt, npt))
    # Add on detector response
    nr_times += det_times
    er_times += det_times
    areas = pmt.sample((evt, npt))
    y = np.asarray(m(timeline, areas, nr_times))
    q = np.asarray(m(timeline_highres, areas, nr_times))
    y += np.random.normal(loc = 0, scale = 3e-3, size = y.shape)
   for a, t0 in zip(areas[:10], nr_times[:10]):
        fancyconfusiontrue.append(npt)
        peaks = []
        thebigpeak = [0]
        thebigcount = [0]
        triplepeaks = []
        timecounter = []
        triplecounter = []
        tryVal = 0
        count = 0
        fig, ax = plt.subplots()
        inty = []
        intt = []
        y = m(timeline, a, t0) + np.random.normal(loc = 0, scale = 1e-3, size = timeline.shape)
        ycml.append(_y)
        sumtml.append(timeline)
        sumnpt.append(npt)
        ### APPEND THINGS TO ARRAYS HERE !!!!
        ybase = np.vstack([ybase, _y])
        nptbase = np.vstack([nptbase, npt])
        # print(ybase.shape)
        # print(nptbase.shape)
        ### APPEND THINGS TO ARRAYS HERE !!!!
        # print(_y)
        for j in range(len(_y)):
            gotdata = 0
            testvar = _y[j]
            trym = 'true'
            tryp = 'true'
            try:
                testvar = _y[j-1]
            except:
                trym = 'null'
                gotdata += 1
            try:
                testvar = _y[j+1]
            except:
                tryp = 'null'
                gotdata += 1
            if trym == 'true':
                if _y[j] > _y[j-1]:
                    gotdata += 1
            if tryp == 'true':
                if _y[j] > _y[j+1]:
                    gotdata += 1
            mean = sum(_y)/len(_y)
            gate = mean + 0.0015
            # if mean < 0.0025:
            #     mean = 0.0025
            if _y[j] > gate:
                inty.append(_y[j])
                intt.append(timeline[j])
            lzmain = -floor(log10(abs(_y[j])))
            lzclean = -floor(log10(abs(mean)))
            if lzmain < lzclean:
                if gotdata > 1:
                    peaks.append(_y[j])
                    timecounter.append(timeline[j])
            else:
                if _y[j] > thebigpeak[0]:
                    thebigpeak[0] = _y[j]
                    thebigcount[0] = timeline[j]
            # if j == (len(_y)-1) and len(peaks) == 0:
            #     peaks.append(thebigpeak[0])
            #     timecounter.append(thebigcount[0])
        areaunderthecurve = np.trapezoid(inty, intt, 1.0, axis=0)
        # print(areaunderthecurve)
        for _a, _t0 in zip(a, t0):
            ax.plot(
                timeline_highres,
                m(timeline_highres, [_a], [_t0])
            )
        ax.plot(
            timeline,
            _y,
            color = "black",
            label = "Detected signal"
        )
        ax.axhline(
            y=gate,
            color='r',
            linestyle='-',
            label = "Threshold"
        )
        ax.scatter(
            timecounter,
            peaks,
            color = "purple",
            label = "Estimated peak"
        )
        plt.fill_between(intt, mean, inty, color='blue', alpha=0.3, label=f"Integrated area ({np.round(areaunderthecurve,2)})")
        if len(peaks) > 3:
            pkfnl = 4
        else:
            pkfnl = len(peaks)
        position1 = npt
        position2 = pkfnl
        print(f"Found {pkfnl}")
        print(f"Was {npt}")
        # confarray[position1][position2] += 1
        # if npt == pkfnl:
        #     wincounter += 1
        # counter += 1
        # if npt == 1:
        #     zcount += 1
        #     if pkfnl == 1:
        #         zwin += 1
        # if npt == 2:
        #     ocount += 1
        #     if pkfnl == 2:
        #         owin += 1
        # if npt == 3:
        #     tcount += 1
        #     if pkfnl == 3:
        #         twin += 1
        # if npt == 4:
        #     hcount += 1
        #     if pkfnl == 4:
        #         hwin += 1
        # npconfusion[npt,pkfnl] += 1
        # fancyconfusionguess.append(pkfnl)
        ax.legend()
        matplotlib.pyplot.close()
#####
# YOUR SAVE DATA IS HERE !!!
#####
savetable = np.append(ybase,nptbase,axis=1)
print("Savetable shape is " + str(savetable.shape))
savetable = np.delete(savetable, 0, axis=0)
print("Savetable shape is " + str(savetable.shape))
print(len(ycml))
print(len(sumnpt))
print(len(sumtml))
if i == 0:
    np.save("EBRNTest",savetable)
else:
    np.save("EBRNTrainBig07",savetable)
print("Saved!")
#####
# YOUR SAVE DATA IS HERE !!!
#####
# for i in range(len(actuallist)):
#     position1 = int(actuallist[i])
#     position2 = int(winnerslist[i])
#     confarray[position1-1][position2-1] += 1
df_cm = pd.DataFrame(confarray, index = [i for i in "1234"],
                columns = [i for i in "1234"])
plt.figure(figsize = (10,7))
ax = sn.heatmap(df_cm, annot=True)
plt.xlabel('Predicted', fontsize = 15)
plt.ylabel('Actual', fontsize = 15)
print(f"Accuracy: {(wincounter/counter)*100}%")
print(f"For one photon: {(zwin/zcount)*100}%")
print(f"For two photons: {(owin/ocount)*100}%")
print(f"For three photons: {(twin/tcount)*100}%")
print(f"For four photons: {(hwin/hcount)*100}%")
np.savetxt("lzpcconfusion.txt", npconfusion, fmt='%d')
# akd.to_parquet(akdconfusion, "lzpcconfusion.parquet")